In [1]:
!pip install -q pandas pyarrow scikit-learn scipy huggingface_hub transformers torch sentencepiece tiktoken protobuf matplotlib tqdm

In [2]:
import os, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

In [6]:
from huggingface_hub import HfApi, hf_hub_download

REPO = "pminervini/HaluEval"
qa_files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset")
            if f.startswith("qa/") and f.endswith(".parquet")]
print("QA parquet files:", qa_files)

path = hf_hub_download(REPO, qa_files[0], repo_type="dataset")
raw = pd.read_parquet(path, engine="pyarrow")

cols = ["knowledge", "question", "right_answer", "hallucinated_answer"]
missing = [c for c in cols if c not in raw.columns]
assert not missing, f"Missing columns: {missing}; found {list(raw.columns)}"

raw = raw[cols].dropna().reset_index(drop=True)
raw = raw.sample(10000, random_state=42).reset_index(drop=True)
raw["record_id"] = raw.index
print(raw.shape)
raw.head(3)

QA parquet files: ['qa/data-00000-of-00001.parquet', 'qa/samples-00000-of-00001.parquet']
(10000, 5)


,knowledge,question,right_answer,hallucinated_answer,record_id
0,He transferred to Wolverhampton Wanderers for...,The manager in which Mark Lazarus clashed with...,1948 and 1964,Mark Lazarus clashed with the manager of Wolve...,0
1,No. 11 Squadron is a Royal Australian Air Forc...,"No. 11 Squadron RAAF was based at what base, 2...",RAAF Base Edinburgh,"RAAF Base Edinburgh, located in Australia.",1
2,"Kim Roi-ha (born November 15, 1965) is a South...",Which movie starring Kim Roi-ha is based on Ko...,Memories of Murder,"Kim Roi-ha starred in ""The Host"".",2


In [7]:
base = raw[["record_id", "question", "knowledge"]]
pos = base.assign(answer=raw["right_answer"], label=0)
neg = base.assign(answer=raw["hallucinated_answer"], label=1)
data = pd.concat([pos, neg], ignore_index=True)

data["answer_words"] = data["answer"].str.split().str.len()
data["knowledge_words"] = data["knowledge"].str.split().str.len()
data["question_words"] = data["question"].str.split().str.len()

print(data.shape)
print(data["label"].value_counts())
data.groupby("label")[["answer_words", "knowledge_words"]].describe().T

(20000, 8)
label
0    10000
1    10000
Name: count, dtype: int64


label                             0             1
answer_words    count  10000.000000  10000.000000
                mean       2.224700     10.986500
                std        1.844926      6.145787
                min        1.000000      1.000000
                25%        1.000000      6.000000
                50%        2.000000      9.000000
                75%        3.000000     14.000000
                max       81.000000     61.000000
knowledge_words count  10000.000000  10000.000000
                mean      55.447400     55.447400
                std       21.597395     21.597395
                min       11.000000     11.000000
                25%       40.000000     40.000000
                50%       52.000000     52.000000
                75%       67.000000     67.000000
                max      256.000000    256.000000

In [8]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(data, groups=data["record_id"]))
train_full, test = data.iloc[tr_idx].copy(), data.iloc[te_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42)
tr2, va2 = next(gss2.split(train_full, groups=train_full["record_id"]))
train, val = train_full.iloc[tr2].copy(), train_full.iloc[va2].copy()

for a, b, n in [(train, val, "train/val"), (train, test, "train/test"), (val, test, "val/test")]:
    assert set(a.record_id).isdisjoint(set(b.record_id)), f"record overlap in {n}"
for name, df in [("train", train), ("val", val), ("test", test)]:
    assert (df.groupby("record_id").size() == 2).all(), f"{name}: a record was split"
    print(f"{name:5s}: {len(df):6d} examples, {df.record_id.nunique():5d} records, hallucinated share = {df.label.mean():.2f}")

train:  14000 examples,  7000 records, hallucinated share = 0.50
val  :   2000 examples,  1000 records, hallucinated share = 0.50
test :   4000 examples,  2000 records, hallucinated share = 0.50


In [9]:
def best_threshold(y_true, score):
    grid = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 197)))
    f1s = [precision_recall_fscore_support(y_true, (score >= t).astype(int), average="binary",
                                           pos_label=1, zero_division=0)[2] for t in grid]
    return float(grid[int(np.argmax(f1s))])

def evaluate(name, y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    return {"model": name, "precision": p, "recall": r, "f1": f}

In [10]:
def make_tfidf_model(C):
    pre = ColumnTransformer([
        ("question",  TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "question"),
        ("knowledge", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000), "knowledge"),
        ("answer",    TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True), "answer"),
    ])
    return Pipeline([("tfidf", pre),
                     ("lr", LogisticRegression(C=C, max_iter=2000, class_weight="balanced"))])

feat_cols = ["question", "knowledge", "answer"]
results_val = {}
for C in [0.1, 1, 10]:
    m = make_tfidf_model(C).fit(train[feat_cols], train["label"])
    s = m.predict_proba(val[feat_cols])[:, 1]
    thr = best_threshold(val["label"], s)
    f1 = evaluate("tfidf", val["label"], (s >= thr).astype(int))["f1"]
    results_val[C] = (f1, thr)
    print(f"C={C:<5} val F1={f1:.4f} (threshold={thr:.3f})")

best_C = max(results_val, key=lambda c: results_val[c][0])
tfidf_thr = results_val[best_C][1]
tfidf_model = make_tfidf_model(best_C).fit(train[feat_cols], train["label"])
test["tfidf_score"] = tfidf_model.predict_proba(test[feat_cols])[:, 1]
test["tfidf_pred"] = (test["tfidf_score"] >= tfidf_thr).astype(int)
val["tfidf_score"] = tfidf_model.predict_proba(val[feat_cols])[:, 1]
print("Chosen C:", best_C, "| threshold:", round(tfidf_thr, 3))

C=0.1   val F1=0.9065 (threshold=0.474)
C=1     val F1=0.9186 (threshold=0.466)
C=10    val F1=0.9210 (threshold=0.429)
Chosen C: 10 | threshold: 0.429


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device).eval()

id2label = {int(k): v.lower() for k, v in nli.config.id2label.items()}
ENT = [k for k, v in id2label.items() if v.startswith("entail")][0]
print("device:", device, "| labels:", id2label, "| entailment index:", ENT)

@torch.no_grad()
def entailment_prob(premises, hypotheses, batch_size=32):
    order = np.argsort([len(p) + len(h) for p, h in zip(premises, hypotheses)])
    out = np.zeros(len(premises), dtype=np.float32)
    for i in tqdm(range(0, len(order), batch_size), desc="NLI scoring"):
        idx = order[i:i + batch_size]
        enc = tok([premises[j] for j in idx], [hypotheses[j] for j in idx],
                  truncation="only_first", max_length=512, padding=True, return_tensors="pt").to(device)
        probs = torch.softmax(nli(**enc).logits, dim=-1)[:, ENT]
        out[idx] = probs.cpu().numpy()
    return out

def nli_scores(df):
    hyp = (df["question"] + " " + df["answer"]).tolist()
    return 1.0 - entailment_prob(df["knowledge"].tolist(), hyp)

cache = f"nli_scores_{NLI_MODEL.split('/')[-1]}_{len(data)}_{42}.parquet"
if os.path.exists(cache):
    cached = pd.read_parquet(cache)
    val["nli_score"] = cached.loc[val.index, "nli_score"].values
    test["nli_score"] = cached.loc[test.index, "nli_score"].values
else:
    val["nli_score"] = nli_scores(val)
    test["nli_score"] = nli_scores(test)
    pd.concat([val["nli_score"], test["nli_score"]]).rename("nli_score").to_frame().to_parquet(cache)

nli_thr = best_threshold(val["label"], val["nli_score"])
test["nli_pred"] = (test["nli_score"] >= nli_thr).astype(int)
print("NLI threshold chosen on validation:", round(nli_thr, 3))

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

... (5 outputs left out)

device: cuda | labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'} | entailment index: 1


NLI scoring:   0%|          | 0/63 [00:00<?, ?it/s]

NLI scoring:   0%|          | 0/125 [00:00<?, ?it/s]

NLI threshold chosen on validation: 0.003


In [12]:
rows = [evaluate("TF-IDF + LogReg", test["label"], test["tfidf_pred"]),
        evaluate("NLI cross-encoder", test["label"], test["nli_pred"])]
metrics = pd.DataFrame(rows).set_index("model").round(4)
display(metrics)

for name, col in [("TF-IDF + LogReg", "tfidf_pred"), ("NLI cross-encoder", "nli_pred")]:
    print(f"\n=== {name} ===")
    print(classification_report(test["label"], test[col], target_names=["supported (0)", "hallucinated (1)"], digits=4))
    cm = confusion_matrix(test["label"], test[col])
    print(pd.DataFrame(cm, index=["true supported", "true hallucinated"], columns=["pred supported", "pred hallucinated"]))

,precision,recall,f1
model,,,
TF-IDF + LogReg,0.9136,0.9300,0.9217
NLI cross-encoder,0.5000,0.9855,0.6634



=== TF-IDF + LogReg ===
                  precision    recall  f1-score   support

   supported (0)     0.9287    0.9120    0.9203      2000
hallucinated (1)     0.9136    0.9300    0.9217      2000

        accuracy                         0.9210      4000
       macro avg     0.9211    0.9210    0.9210      4000
    weighted avg     0.9211    0.9210    0.9210      4000

                   pred supported  pred hallucinated
true supported               1824                176
true hallucinated             140               1860

=== NLI cross-encoder ===
                  precision    recall  f1-score   support

   supported (0)     0.5000    0.0145    0.0282      2000
hallucinated (1)     0.5000    0.9855    0.6634      2000

        accuracy                         0.5000      4000
       macro avg     0.5000    0.5000    0.3458      4000
    weighted avg     0.5000    0.5000    0.3458      4000

                   pred supported  pred hallucinated
true supported                 29 

In [13]:
def show_examples(df, pred_col, score_col, kind, n=5, width=110):
    if kind == "FP":
        sub = df[(df["label"] == 0) & (df[pred_col] == 1)]
    else:
        sub = df[(df["label"] == 1) & (df[pred_col] == 0)]
    print(f"##### {kind}: {len(sub)} cases in test; showing {min(n, len(sub))} #####\n")
    for _, r in sub.sample(min(n, len(sub)), random_state=42).iterrows():
        print(f"record {r.record_id} | score={r[score_col]:.3f} | answer_words={r.answer_words} | knowledge_words={r.knowledge_words}")
        print("  Q:", r.question)
        print("  K:", textwrap.shorten(r.knowledge, 450))
        print("  A:", r.answer)
        print("  gold:", "hallucinated" if r.label else "supported", "\n")

print("================ TF-IDF ================")
show_examples(test, "tfidf_pred", "tfidf_score", "FP")
show_examples(test, "tfidf_pred", "tfidf_score", "FN")

================ TF-IDF ================
##### FP: 176 cases in test; showing 5 #####

record 1544 | score=1.000 | answer_words=2 | knowledge_words=38
  Q: What is the nickname of Daulis Prescott's brother?
  K: His brother Breidis Prescott is also a professional boxer. In 2008, Prescott first became known on the world stage when he upset heavily favoured and then-undefeated Olympian Amir Khan, which gave rise to Prescott's nickname of "The Khanqueror".
  A: "The Khanqueror".
  gold: supported 

record 3314 | score=1.000 | answer_words=4 | knowledge_words=66
  Q: Adrian Sutil scored this F1 teams only point ?
  K: Adrian Sutil scored Spyker's first and only point by finishing 9th and being promoted to 8th after the race.The Spyker F1 Team, known as the Etihad Aldar Spyker F1 Team for sponsorship reasons was a Formula One team that competed in the 2007 Formula One World Championship, and was created by Spyker Cars after their buyout of the short-lived Midland F1 (formerly Jordan Grand P

In [14]:
print("================ NLI cross-encoder ================")
show_examples(test, "nli_pred", "nli_score", "FP")
show_examples(test, "nli_pred", "nli_score", "FN")

================ NLI cross-encoder ================
##### FP: 1971 cases in test; showing 5 #####

record 3767 | score=0.009 | answer_words=2 | knowledge_words=59
  Q: Who wrote the musical in which Edwin Gagiano won a Best Actor Award? 
  K: He is regarded as one of South Africa’s highest-earning actors by The New Age and has won several awards for his achievements, including Best Actor Award for his role as Alf Bueller in the show "Back to the 80s".Back to the 80's [sic] is a musical written by Neil Gooding with the original musical adaption made by Scott Copeman.
  A: Neil Gooding
  gold: supported 

record 3842 | score=0.992 | answer_words=2 | knowledge_words=111
  Q: "Bass Down Low" is a single made by Dev as a follow up to a Far East Movements song from what album?
  K: "Bass Down Low" is the debut single by American recording artist Dev, written and performed by Dev and The Cataracs. Initially, the song was made for The Cataracs with Dev as the featured act, but she was made the